In [ ]:
import sys, json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.geometry import (pca, best_parabola_in_subspace, best_chirp_in_subspace,
                          best_ellipse_in_subspace, neumann_2d_mode,
                          fit_2d_modes_multipc)

BG     = '#F5F0EB'
AX_BG  = '#FAFAF8'
BORDER = '#C8C0B8'
TEXT   = '#2C2C2C'
MUTED  = '#9E9690'
RED    = '#C0392B'
BLUE   = '#2A6EBB'
ORANGE = '#E67E22'
MODE_COLORS = {1: '#2A6EBB', 2: '#8E44AD', 3: '#E67E22'}

FIGS_DIR    = ROOT / 'figures';  FIGS_DIR.mkdir(exist_ok=True)
RESULTS_DIR = ROOT / 'results'
N_LAYERS    = 19

def ax_style(ax, title='', xlabel='', ylabel='', grid=True):
    ax.set_facecolor(AX_BG)
    for sp in ax.spines.values():
        sp.set_edgecolor(BORDER); sp.set_linewidth(0.8)
    ax.tick_params(colors=TEXT, labelsize=8.5, length=3)
    ax.xaxis.label.set_color(TEXT); ax.yaxis.label.set_color(TEXT)
    ax.title.set_color(TEXT)
    if title:  ax.set_title(title, fontsize=10, pad=5, fontweight='semibold')
    if xlabel: ax.set_xlabel(xlabel, fontsize=9)
    if ylabel: ax.set_ylabel(ylabel, fontsize=9)
    if grid:   ax.grid(True, color=BORDER, lw=0.5, alpha=0.7, ls='--')

def heatmap_panel(ax, vecs, words, title=''):
    vecs  = np.array(vecs, dtype=float)
    norms = np.linalg.norm(vecs, axis=1, keepdims=True)
    G     = (vecs / (norms + 1e-12)) @ (vecs / (norms + 1e-12)).T
    vmax  = np.abs(G).max() * 0.97
    im    = ax.imshow(G, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='equal')
    n     = len(words)
    ax.set_xticks(range(n)); ax.set_xticklabels(words, rotation=40, ha='right', fontsize=7.5)
    ax.set_yticks(range(n)); ax.set_yticklabels(words, fontsize=7.5)
    divider = make_axes_locatable(ax)
    cax  = divider.append_axes('right', size='5%', pad=0.05)
    cbar = plt.colorbar(im, cax=cax)
    cbar.ax.tick_params(colors=TEXT, labelsize=7.5)
    cbar.ax.set_title('cos sim', color=TEXT, fontsize=8, pad=4)
    cbar.outline.set_edgecolor(BORDER); cbar.outline.set_linewidth(0.8)
    if title: ax.set_title(title, fontsize=9.5, color=TEXT, pad=4)
    ax.tick_params(colors=TEXT, labelsize=7.5)
    for sp in ax.spines.values():
        sp.set_edgecolor(BORDER); sp.set_linewidth(0.8)

def parabola_panel(ax, xr, yr, coeffs, words, ranks, source='', label_size=6.0, source_ha='right'):
    xr, yr, ranks = np.array(xr), np.array(yr), np.array(ranks)
    coeffs = list(coeffs)
    # Sign-normalise: flip y so the parabola opens upward
    flipped_y = coeffs[0] < 0
    if flipped_y:
        yr     = -yr
        coeffs = [-c for c in coeffs]
    x_dense = np.linspace(xr.min(), xr.max(), 200)
    ax.plot(x_dense, np.polyval(coeffs, x_dense), color='grey', lw=1.4, ls='--', zorder=2, alpha=0.45)
    ax.scatter(xr, yr, c=ranks, cmap='RdYlGn', s=55, zorder=4,
               edgecolors='white', lw=0.5, vmin=ranks.min(), vmax=ranks.max())
    for i, w in enumerate(words):
        ax.annotate(w, (xr[i], yr[i]), fontsize=label_size, color=TEXT,
                    ha='center', va='bottom', xytext=(0, 4), textcoords='offset points')
    if source:
        x_src = {'left': 0.03, 'right': 0.97}.get(source_ha, 0.97)
        ax.annotate(source, xy=(x_src, 0.04), xycoords='axes fraction',
                    fontsize=7.5, color=MUTED, ha=source_ha, style='italic')
    return flipped_y

## Periodic BC

In [ ]:
with open(RESULTS_DIR / 'periodic.json') as f:
    data_c = json.load(f)
sr_c = data_c['glove']['scale_results']

_npz = RESULTS_DIR / 'llm_periodic.npz'
llm  = np.load(str(_npz), allow_pickle=True)
acts_c = llm['activations'].item()
meta_c = llm['meta'].item()

from src.concepts import PERIODIC_SCALES

GLOVE_KEYS_C = ['months',           'days_of_week',           'compass']
LLM_KEYS_C   = ['periodic_months',  'periodic_days_of_week',  'periodic_compass']
LABELS_C     = ['Months',            'Days of week',           'Compass']

GREEN = '#27AE60'

def ellipse_panel(ax, coords, phases, words, source='', label_size=6.0, source_ha='right', show_fit=True):
    """Plot the best-fit ellipse and data points colored by cyclic phase."""
    phases = np.asarray(phases, dtype=float)
    res = best_ellipse_in_subspace(coords, phases, words)
    bf  = res['best_fit']
    i, j = res['best_pair']

    x_pts = coords[:, i]
    y_pts = coords[:, j]

    if show_fit:
        xc = np.array(bf['x_coeffs'])
        yc = np.array(bf['y_coeffs'])
        n  = res['best_harmonic']
        t_dense = np.linspace(0, 1, 300)
        theta_d = 2 * np.pi * n * t_dense
        A_d     = np.column_stack([np.cos(theta_d), np.sin(theta_d), np.ones(300)])
        ax.plot(A_d @ xc, A_d @ yc, color='grey', lw=1.4, ls='--', zorder=2, alpha=0.45)

    ax.scatter(x_pts, y_pts, c=phases, cmap='RdYlGn', s=55, zorder=4,
               edgecolors='white', lw=0.5, vmin=0, vmax=1)
    for k_w, w in enumerate(words):
        ax.annotate(w, (x_pts[k_w], y_pts[k_w]), fontsize=label_size, color=TEXT,
                    ha='center', va='bottom', xytext=(0, 4), textcoords='offset points')
    if source:
        x_src = {'left': 0.03, 'right': 0.97}.get(source_ha, 0.97)
        ax.annotate(source, xy=(x_src, 0.04), xycoords='axes fraction',
                    fontsize=7.5, color=MUTED, ha=source_ha, style='italic')
    return i, j

print('Finding best Gemma layers (periodic, skip embedding layer) ...')
best_layers_c, gemma_coords_c, gemma_res_c = {}, {}, {}
for llm_key in LLM_KEYS_C:
    scale_key = llm_key.replace('periodic_', '')
    words  = meta_c[llm_key]['words']
    N_w    = len(words)
    phases = np.arange(N_w) / N_w
    layer_r2 = []
    for layer in range(1, N_LAYERS):
        vecs = acts_c[llm_key][str(layer)]
        k    = min(N_w - 1, 6)
        coords, _, _ = pca(vecs, n_components=k)
        res = best_ellipse_in_subspace(coords, phases, words)
        layer_r2.append(res['best_r2'])
    bl = int(np.argmax(layer_r2)) + 1
    best_layers_c[llm_key] = bl
    vecs = acts_c[llm_key][str(bl)]
    k    = min(N_w - 1, 6)
    coords, variances, _ = pca(vecs, n_components=k)
    gemma_coords_c[llm_key] = coords
    gemma_res_c[llm_key]    = best_ellipse_in_subspace(coords, phases, words)
    pi, pj = gemma_res_c[llm_key]['best_pair']
    vf = float((variances[pi] + variances[pj]) / variances.sum())
    print(f'  {llm_key:<28} layer={bl}  φ{pi+1}–φ{pj+1}  R²={layer_r2[bl-1]:.3f}  var={vf:.1%}')

In [ ]:
fig_c = plt.figure(figsize=(11, 6.0), facecolor=BG)
gs_c  = gridspec.GridSpec(2, 4, figure=fig_c,
                           left=0.09, right=0.97, top=0.92, bottom=0.12,
                           wspace=0.4, hspace=0.35,
                           width_ratios=[0.9, 0.9, 0.9, 0.9])

# GloVe row
ax_c_hm0 = fig_c.add_subplot(gs_c[0, 0])
_s0 = sr_c['months']
_k  = min(len(_s0['words']) - 1, 6)
_hm_coords, _, _ = pca(np.array(_s0['embeddings']), n_components=_k)
heatmap_panel(ax_c_hm0, _hm_coords, _s0['words'], title='GloVe')

print('GloVe periodic fits:')
panels_c_row0 = []
for col, (gkey, label) in enumerate(zip(GLOVE_KEYS_C, LABELS_C), start=1):
    ax = fig_c.add_subplot(gs_c[0, col])
    panels_c_row0.append(ax)
    s      = sr_c[gkey]
    words  = s['words']
    phases = np.array(s['phases'])
    k      = min(len(words) - 1, 6)
    coords, variances, _ = pca(np.array(s['embeddings']), n_components=k)
    show = gkey not in ('months', 'days_of_week')
    pi, pj = ellipse_panel(ax, coords, phases, words, source='GloVe 300d', show_fit=show)
    vf = float((variances[pi] + variances[pj]) / variances.sum())
    print(f'  {label:<15} φ{pi+1}–φ{pj+1}  var={vf:.1%}')
    ax_style(ax, title=label,
             xlabel=fr'$\phi_{{{pi+1}}}$', ylabel=fr'$\phi_{{{pj+1}}}$')

# Gemma row
ax_c_hm1 = fig_c.add_subplot(gs_c[1, 0])
heatmap_panel(ax_c_hm1, gemma_coords_c['periodic_months'],
              meta_c['periodic_months']['words'], title='Gemma 2B')

print('Gemma periodic fits:')
panels_c_row1 = []
for col, (llm_key, label) in enumerate(zip(LLM_KEYS_C, LABELS_C), start=1):
    ax  = fig_c.add_subplot(gs_c[1, col])
    panels_c_row1.append(ax)
    bl     = best_layers_c[llm_key]
    words  = meta_c[llm_key]['words']
    phases = np.arange(len(words)) / len(words)
    pi, pj = ellipse_panel(ax, gemma_coords_c[llm_key], phases, words,
                           source=f'layer {bl}', source_ha='left')
    _gemma_vars = pca(acts_c[llm_key][str(bl)], n_components=min(len(words)-1, 6))[1]
    vf_l = float((_gemma_vars[pi] + _gemma_vars[pj]) / _gemma_vars.sum())
    print(f'  {label:<15} layer={bl}  φ{pi+1}–φ{pj+1}  var={vf_l:.1%}')
    ax_style(ax, title=label,
             xlabel=fr'$\phi_{{{pi+1}}}$', ylabel=fr'$\phi_{{{pj+1}}}$')

# Shift panels right to give heatmap+cbar column more room
fig_c.canvas.draw()
shift = 0.05
for ax in panels_c_row0 + panels_c_row1:
    pos = ax.get_position()
    ax.set_position([pos.x0 + shift, pos.y0, pos.width * 0.95, pos.height])

# Align panel heights to their row's heatmap
fig_c.canvas.draw()
for hm_ax, panels in [(ax_c_hm0, panels_c_row0), (ax_c_hm1, panels_c_row1)]:
    hm_pos = hm_ax.get_position()
    for ax in panels:
        pos = ax.get_position()
        ax.set_position([pos.x0, hm_pos.y0, pos.width, hm_pos.height])

# Row labels
fig_c.canvas.draw()
for panels, lbl in [(panels_c_row0, 'GloVe'), (panels_c_row1, 'Gemma 2B')]:
    pos = panels[0].get_position()
    fig_c.text(pos.x0 - 0.052, (pos.y0 + pos.y1) / 2, lbl,
               color=TEXT, fontsize=11, fontweight='bold',
               va='center', ha='right', rotation=90)

fig_c.patch.set_facecolor(BG)
path = FIGS_DIR / 'periodic_main.png'
plt.savefig(path, bbox_inches='tight', facecolor=BG, dpi=300)
plt.show()
print(f'Saved {path.name}')

## Neumann BC

In [ ]:
with open(RESULTS_DIR / 'neumann.json') as f:
    data_a = json.load(f)
sr = data_a['glove']['scale_results']

_npz = RESULTS_DIR / 'llm_neumann.npz'
llm  = np.load(str(_npz), allow_pickle=True)
acts = llm['activations'].item()
meta = llm['meta'].item()

GLOVE_KEYS = ['quality',         'temperature',          'emotion_valence']
LLM_KEYS   = ['ordinal_quality', 'ordinal_temperature',  'ordinal_emotion_valence']
LABELS_A   = ['Quality',         'Temperature',          'Emotion valence']

print('Finding best Gemma layers (Neumann, n_pc=10, exhaustive, skip embedding layer) ...')
best_layers_a, gemma_coords_a, gemma_res_a, gemma_vars_a = {}, {}, {}, {}
for llm_key in LLM_KEYS:
    words = meta[llm_key]['words']
    n_pc = min(len(words) - 1, 10)
    layer_r2 = []
    for layer in range(1, N_LAYERS):
        vecs = acts[llm_key][str(layer)]
        coords, variances, _ = pca(vecs, n_components=n_pc)
        res = best_parabola_in_subspace(coords, words, variances=variances)
        layer_r2.append(res['best_r2'])
    bl = int(np.argmax(layer_r2)) + 1
    best_layers_a[llm_key] = bl
    vecs = acts[llm_key][str(bl)]
    coords, variances, _ = pca(vecs, n_components=n_pc)
    res = best_parabola_in_subspace(coords, words, variances=variances)
    gemma_coords_a[llm_key] = coords
    gemma_vars_a[llm_key]   = variances
    gemma_res_a[llm_key]    = res
    pi, pj = res['best_pair']
    vf = res['best_fit'].get('var_fraction', 0)
    print(f'  {llm_key:<28} layer={bl}  φ{pi+1}–φ{pj+1}  R²={layer_r2[bl-1]:.3f}  var={vf:.1%}')

In [ ]:
fig_a = plt.figure(figsize=(11, 6.0), facecolor=BG)
gs_a  = gridspec.GridSpec(2, 4, figure=fig_a,
                           left=0.09, right=0.97, top=0.92, bottom=0.12,
                           wspace=0.4, hspace=0.35,
                           width_ratios=[0.9, 0.9, 0.9, 0.9])

# GloVe row
ax_a_hm0 = fig_a.add_subplot(gs_a[0, 0])
_k = min(len(sr['quality']['words']) - 1, 10)
_hm_coords, _, _ = pca(np.array(sr['quality']['embeddings']), n_components=_k)
heatmap_panel(ax_a_hm0, _hm_coords, sr['quality']['words'], title='GloVe')

print('GloVe Neumann fits:')
panels_a_row0 = []
for col, (gkey, label) in enumerate(zip(GLOVE_KEYS, LABELS_A), start=1):
    ax = fig_a.add_subplot(gs_a[0, col])
    panels_a_row0.append(ax)
    s    = sr[gkey]
    n_pc = min(len(s['words']) - 1, 10)
    coords, variances, _ = pca(np.array(s['embeddings']), n_components=n_pc)
    bres = best_parabola_in_subspace(coords, s['words'], variances=variances)
    bf   = bres['best_fit']
    pi, pj = bres['best_pair']
    vf = bf.get('var_fraction', 0)
    print(f'  {label:<20} φ{pi+1}–φ{pj+1}  R²={bres["best_r2"]:.3f}  var={vf:.1%}')
    flip = parabola_panel(ax, bf['x_rot'], bf['y_rot'], bf['coeffs'],
                          s['words'], s['ordinal_ranks'], source='GloVe 300d')
    ax_style(ax, title=label, xlabel=fr'$\phi_{{{pi+1}}}$',
             ylabel=fr'$-\phi_{{{pj+1}}}$' if flip else fr'$\phi_{{{pj+1}}}$')

# Gemma row
ax_a_hm1 = fig_a.add_subplot(gs_a[1, 0])
heatmap_panel(ax_a_hm1, gemma_coords_a['ordinal_quality'],
              meta['ordinal_quality']['words'], title='Gemma 2B')

print('Gemma Neumann fits:')
panels_a_row1 = []
for col, (llm_key, label) in enumerate(zip(LLM_KEYS, LABELS_A), start=1):
    ax  = fig_a.add_subplot(gs_a[1, col])
    panels_a_row1.append(ax)
    gres = gemma_res_a[llm_key]
    bf   = gres['best_fit']
    bl   = best_layers_a[llm_key]
    pi, pj = gres['best_pair']
    vf = bf.get('var_fraction', 0)
    print(f'  {label:<20} layer={bl}  φ{pi+1}–φ{pj+1}  R²={gres["best_r2"]:.3f}  var={vf:.1%}')
    words = meta[llm_key]['words']
    flip = parabola_panel(ax, bf['x_rot'], bf['y_rot'], bf['coeffs'],
                          words, list(range(len(words))),
                          source=f'layer {bl}', source_ha='left')
    ax_style(ax, title=label, xlabel=fr'$\phi_{{{pi+1}}}$',
             ylabel=fr'$-\phi_{{{pj+1}}}$' if flip else fr'$\phi_{{{pj+1}}}$')

# Shift panels right to give heatmap+cbar column more room
fig_a.canvas.draw()
shift = 0.05
for ax in panels_a_row0 + panels_a_row1:
    pos = ax.get_position()
    ax.set_position([pos.x0 + shift, pos.y0, pos.width * 0.95, pos.height])

# Align panel heights to their row's heatmap
fig_a.canvas.draw()
for hm_ax, panels in [(ax_a_hm0, panels_a_row0), (ax_a_hm1, panels_a_row1)]:
    hm_pos = hm_ax.get_position()
    for ax in panels:
        pos = ax.get_position()
        ax.set_position([pos.x0, hm_pos.y0, pos.width, hm_pos.height])

# Row labels
fig_a.canvas.draw()
for panels, lbl in [(panels_a_row0, 'GloVe'), (panels_a_row1, 'Gemma 2B')]:
    pos = panels[0].get_position()
    fig_a.text(pos.x0 - 0.052, (pos.y0 + pos.y1) / 2, lbl,
               color=TEXT, fontsize=11, fontweight='bold',
               va='center', ha='right', rotation=90)

fig_a.patch.set_facecolor(BG)
path = FIGS_DIR / 'neumann_main.png'
plt.savefig(path, bbox_inches='tight', facecolor=BG, dpi=300)
plt.show()
print(f'Saved {path.name}')

## Log scale

In [ ]:
with open(RESULTS_DIR / 'log_scale.json') as f:
    data_b = json.load(f)
sr_b = data_b['glove']['scale_results']

_npz = RESULTS_DIR / 'llm_log_scale.npz'
llm  = np.load(str(_npz), allow_pickle=True)
acts = llm['activations'].item()
meta = llm['meta'].item()

GLOVE_KEYS_B = ['storage_full', 'time',         'money']
LLM_KEYS_B   = ['scale_storage', 'scale_time',  'scale_money']
LABELS_B     = ['Storage',        'Time',         'Money']
XLABEL_RAW   = r'norm. log-pos. $u$'

def chirp_panel(ax, u, coords, words, source='', label_size=6.0,
                    source_ha='right', sigma_clip=2.0, extra_modes=()):
    u, coords = np.array(u, dtype=float), np.array(coords, dtype=float)

    sin1 = np.sin(np.pi / 2 * u)
    cos1 = np.cos(np.pi / 2 * u)
    X3   = np.column_stack([sin1, cos1, np.ones(len(u))])
    best_r2, best_pc = -np.inf, 0
    for i in range(coords.shape[1]):
        pc = coords[:, i]
        w3, _, _, _ = np.linalg.lstsq(X3, pc, rcond=None)
        fitted = X3 @ w3
        r2 = 1.0 - np.sum((pc - fitted)**2) / np.sum((pc - pc.mean())**2)
        if r2 > best_r2:
            best_r2, best_pc = r2, i
    pc_vals = coords[:, best_pc]

    mask = np.ones(len(u), dtype=bool)
    wf   = np.array([1.0, 0.0, 0.0])
    for _ in range(4):
        X_in = np.column_stack([sin1[mask], cos1[mask], np.ones(mask.sum())])
        wf, _, _, _ = np.linalg.lstsq(X_in, pc_vals[mask], rcond=None)
        res  = np.abs(pc_vals - (wf[0]*sin1 + wf[1]*cos1 + wf[2]))
        mad  = np.median(res[mask])
        new_mask = res <= sigma_clip * (mad + 1e-12)
        if new_mask.sum() == mask.sum() or new_mask.sum() < 3:
            break
        mask = new_mask

    if wf[0] < wf[1]:
        pc_vals = -pc_vals
        wf = -wf
        flipped = True
    else:
        flipped = False

    u_d      = np.linspace(0, 1, 300)
    pc_curve = wf[0]*np.sin(np.pi/2*u_d) + wf[1]*np.cos(np.pi/2*u_d) + wf[2]

    ax.scatter(u, pc_vals, c=u, cmap='RdYlGn', s=55, zorder=4,
               edgecolors='white', lw=0.5, vmin=0, vmax=1)
    ax.plot(u_d, pc_curve, color='grey', lw=1.4, ls='--', zorder=2, alpha=0.45)

    for em in extra_modes:
        sin_em = np.sin(em * np.pi / 2 * u)
        cos_em = np.cos(em * np.pi / 2 * u)
        mask_em = np.ones(len(u), dtype=bool)
        we = np.array([1.0, 0.0, 0.0])
        for _ in range(6):
            X_em = np.column_stack([sin_em[mask_em], cos_em[mask_em], np.ones(mask_em.sum())])
            we, _, _, _ = np.linalg.lstsq(X_em, pc_vals[mask_em], rcond=None)
            res_em = np.abs(pc_vals - (we[0]*sin_em + we[1]*cos_em + we[2]))
            mad_em = np.median(res_em[mask_em])
            new_em = res_em <= 1.5 * (mad_em + 1e-12)
            if new_em.sum() == mask_em.sum() or new_em.sum() < 3:
                break
            mask_em = new_em
        em_curve = we[0]*np.sin(em*np.pi/2*u_d) + we[1]*np.cos(em*np.pi/2*u_d) + we[2]
        ax.plot(u_d, em_curve, color='grey', lw=1.4, ls='--', zorder=2, alpha=0.30)

    for i, word in enumerate(words):
        ax.annotate(word, (u[i], pc_vals[i]), fontsize=label_size, color=TEXT,
                    ha='center', va='bottom', xytext=(0, 4), textcoords='offset points')
    if source:
        x_src = {'left': 0.03, 'right': 0.97, 'center': 0.5}.get(source_ha, 0.97)
        ax.annotate(source, xy=(x_src, 0.04), xycoords='axes fraction',
                    fontsize=7.5, color=MUTED, ha=source_ha, style='italic')

    prefix = '-' if flipped else ''
    ax.set_ylabel(fr'${prefix}\phi_{{{best_pc+1}}}$', color=TEXT, fontsize=10)
    return best_pc

def _best_single_pc_r2(coords, u):
    sin1 = np.sin(np.pi / 2 * u)
    cos1 = np.cos(np.pi / 2 * u)
    X3   = np.column_stack([sin1, cos1, np.ones(len(u))])
    best, best_idx = -np.inf, 0
    for i in range(coords.shape[1]):
        pc = coords[:, i]
        w, _, _, _ = np.linalg.lstsq(X3, pc, rcond=None)
        fitted = X3 @ w
        r2 = 1.0 - np.sum((pc - fitted)**2) / np.sum((pc - pc.mean())**2)
        if r2 > best:
            best, best_idx = r2, i
    return best, best_idx

print('Finding best Gemma layers (log scale, skip embedding layer) ...')
best_layers_b, gemma_coords_b, gemma_vars_b = {}, {}, {}
for llm_key in LLM_KEYS_B:
    words  = meta[llm_key]['words']
    log10v = np.log10(np.array(meta[llm_key]['values'], dtype=float))
    u      = (log10v - log10v.min()) / (log10v.max() - log10v.min())
    layer_r2 = []
    for layer in range(1, N_LAYERS):
        vecs = acts[llm_key][str(layer)]
        k    = min(len(words) - 1, 6)
        coords, _, _ = pca(vecs, n_components=k)
        layer_r2.append(_best_single_pc_r2(coords, u)[0])
    bl = int(np.argmax(layer_r2)) + 1
    best_layers_b[llm_key] = bl
    vecs = acts[llm_key][str(bl)]
    k    = min(len(words) - 1, 6)
    coords, variances, _ = pca(vecs, n_components=k)
    gemma_coords_b[llm_key] = coords
    gemma_vars_b[llm_key] = variances
    _, best_pc = _best_single_pc_r2(coords, u)
    vf = float(variances[best_pc] / variances.sum())
    print(f'  {llm_key:<15} layer={bl}  φ{best_pc+1}  R²={layer_r2[bl-1]:.3f}  var={vf:.1%}')

In [ ]:
fig_b = plt.figure(figsize=(11, 6.0), facecolor=BG)
gs_b  = gridspec.GridSpec(2, 4, figure=fig_b,
                           left=0.09, right=0.97, top=0.92, bottom=0.12,
                           wspace=0.4, hspace=0.3,
                           width_ratios=[0.9, 0.9, 0.9, 0.9])

# GloVe row
ax_b_hm0 = fig_b.add_subplot(gs_b[0, 0])
_k = min(len(sr_b['time']['words']) - 1, 6)
_hm_coords, _, _ = pca(np.array(sr_b['time']['embeddings']), n_components=_k)
heatmap_panel(ax_b_hm0, _hm_coords, sr_b['time']['words'], title='GloVe')

print('GloVe log-scale fits:')
panels_b_row0 = []
for col, (gkey, llm_key, label) in enumerate(
        zip(GLOVE_KEYS_B, LLM_KEYS_B, LABELS_B), start=1):
    ax = fig_b.add_subplot(gs_b[0, col])
    panels_b_row0.append(ax)

    s      = sr_b[gkey]
    log10v = np.array(s['log10_values'])
    k      = min(len(s['words']) - 1, 6)
    coords, variances, _ = pca(np.array(s['embeddings']), n_components=k)

    gemma_ws = set(meta[llm_key]['words'])
    keep     = [i for i, w in enumerate(s['words']) if w in gemma_ws]
    words_g  = [s['words'][i] for i in keep]
    coords_g = coords[keep]
    log10g   = log10v[keep]
    u_g      = (log10g - log10g.min()) / (log10g.max() - log10g.min())

    _, best_pc_g = _best_single_pc_r2(coords_g, u_g)
    vf_g = float(variances[best_pc_g] / variances.sum())
    print(f'  {label:<15} φ{best_pc_g+1}  var={vf_g:.1%}')

    chirp_panel(ax, u_g, coords_g, words_g, source='GloVe 300d')
    ax_style(ax, title=label, xlabel=XLABEL_RAW, ylabel=None)

# Gemma row
ax_b_hm1 = fig_b.add_subplot(gs_b[1, 0])
heatmap_panel(ax_b_hm1, gemma_coords_b['scale_time'],
              meta['scale_time']['words'], title='Gemma 2B')

print('Gemma log-scale fits:')
panels_b_row1 = []
for col, (llm_key, label) in enumerate(zip(LLM_KEYS_B, LABELS_B), start=1):
    ax  = fig_b.add_subplot(gs_b[1, col])
    panels_b_row1.append(ax)
    bl    = best_layers_b[llm_key]
    words = meta[llm_key]['words']
    log10v = np.log10(np.array(meta[llm_key]['values'], dtype=float))
    u      = (log10v - log10v.min()) / (log10v.max() - log10v.min())
    _, best_pc_l = _best_single_pc_r2(gemma_coords_b[llm_key], u)
    vf_l = float(gemma_vars_b[llm_key][best_pc_l] / gemma_vars_b[llm_key].sum())
    print(f'  {label:<15} layer={bl}  φ{best_pc_l+1}  var={vf_l:.1%}')
    chirp_panel(ax, u, gemma_coords_b[llm_key], words,
                    source=f'layer {bl}', source_ha='center')
    ax_style(ax, title=label, xlabel=XLABEL_RAW, ylabel=None)

# Shift panels right to give heatmap+cbar column more room
fig_b.canvas.draw()
shift = 0.05
for ax in panels_b_row0 + panels_b_row1:
    pos = ax.get_position()
    ax.set_position([pos.x0 + shift, pos.y0, pos.width * 0.95, pos.height])

# Align panel heights to their row's heatmap
fig_b.canvas.draw()
for hm_ax, panels in [(ax_b_hm0, panels_b_row0), (ax_b_hm1, panels_b_row1)]:
    hm_pos = hm_ax.get_position()
    for ax in panels:
        pos = ax.get_position()
        ax.set_position([pos.x0, hm_pos.y0, pos.width, hm_pos.height])

# Row labels
fig_b.canvas.draw()
for panels, lbl in [(panels_b_row0, 'GloVe'), (panels_b_row1, 'Gemma 2B')]:
    pos = panels[0].get_position()
    fig_b.text(pos.x0 - 0.052, (pos.y0 + pos.y1) / 2, lbl,
               color=TEXT, fontsize=11, fontweight='bold',
               va='center', ha='right', rotation=90)

fig_b.patch.set_facecolor(BG)
path = FIGS_DIR / 'log_scale_main.png'
plt.savefig(path, bbox_inches='tight', facecolor=BG, dpi=300)
plt.show()
print(f'Saved {path.name}')

## 2-D Neumann BC (Emotion Circumplex)

Each word is placed at its psychologically normed coordinate $(u, v)$ in valence–arousal space.
For a given eigenmode $\phi_{m,n}(u,v) = \cos(m\pi u)\cos(n\pi v)$, we compute a **target vector** $t_i = \phi_{m,n}(u_i, v_i)$ for each word, then find the linear combination of the top 10 PCs that best reconstructs it via ordinary least squares:

$$\hat{t}_i = w_1\,\mathrm{PC}_{i,1} + w_2\,\mathrm{PC}_{i,2} + \cdots + w_{10}\,\mathrm{PC}_{i,10} + b$$

The dot colour in each panel is this predicted value $\hat{t}_i$ (the "PC projection"), and $R^2$ measures how much of the mode's variance is explained. The background tint shows the theoretical mode shape for visual comparison.

In [ ]:
# ── Load 2-D Neumann data ────────────────────────────────────────────────────

_npz_d    = RESULTS_DIR / 'llm_neumann_2d.npz'
_ld      = np.load(str(_npz_d), allow_pickle=True)
acts_d   = _ld['activations'].item()
meta_d   = _ld['meta'].item()
_ec      = meta_d['emotion_circumplex']
llm_words_d   = _ec['words']
llm_valence_d = np.array(_ec['valence'])
llm_arousal_d = np.array(_ec['arousal'])
N_LAYERS_D    = len(acts_d['emotion_circumplex'])

k_fit = 10
best_sum, best_layer_d = -np.inf, 1
layer_mode_data_d = {}
for layer in range(1, N_LAYERS_D):
    vecs = acts_d['emotion_circumplex'][str(layer)]
    coords, variances, _ = pca(vecs, n_components=k_fit)
    mr = fit_2d_modes_multipc(coords, llm_valence_d, llm_arousal_d, max_m=3, max_n=3)
    layer_mode_data_d[layer] = {'coords': coords, 'vars': variances, 'mode_res': mr}
    s = mr[(1,0)]['r2'] + mr[(0,1)]['r2']
    if s > best_sum:
        best_sum, best_layer_d = s, layer

coords_best_d = layer_mode_data_d[best_layer_d]['coords']
vars_best_d   = layer_mode_data_d[best_layer_d]['vars']
mr_best_d     = layer_mode_data_d[best_layer_d]['mode_res']
total_var_d   = vars_best_d.sum()
print(f'Gemma 2 27B: best layer = {best_layer_d}  '
      f'(1,0) R²={mr_best_d[(1,0)]["r2"]:.3f}  '
      f'(0,1) R²={mr_best_d[(0,1)]["r2"]:.3f}  '
      f'(1,1) R²={mr_best_d[(1,1)]["r2"]:.3f}')

# ── Overlay figure ───────────────────────────────────────────────────────────

def _bg_cmap():
    """Warm tint: dark tan ↔ base beige ↔ warm golden cream."""
    dark  = np.array([0.78, 0.72, 0.62])
    base  = np.array([0.96, 0.94, 0.91])
    light = np.array([0.98, 0.94, 0.82])
    cdict = {'red':   [(0, dark[0], dark[0]),  (0.5, base[0], base[0]), (1, light[0], light[0])],
             'green': [(0, dark[1], dark[1]),  (0.5, base[1], base[1]), (1, light[1], light[1])],
             'blue':  [(0, dark[2], dark[2]),  (0.5, base[2], base[2]), (1, light[2], light[2])]}
    return LinearSegmentedColormap('sand_tint', cdict, N=256)

overlay_modes = [(1, 0), (1, 1)]
bg_cmap = _bg_cmap()

fig_d = plt.figure(figsize=(11, 5.0), facecolor=BG)
gs_d  = gridspec.GridSpec(1, 3, figure=fig_d,
                           left=0.06, right=0.90, top=0.94, bottom=0.10,
                           wspace=0.3, width_ratios=[1, 1, 0.05])

u_grid = np.linspace(-0.1, 1.1, 180)
v_grid = np.linspace(-0.1, 1.1, 180)
U, V = np.meshgrid(u_grid, v_grid)

shared_vmax = 0
for (m, n) in overlay_modes:
    p = np.abs(np.array(mr_best_d[(m, n)]['pred']))
    shared_vmax = max(shared_vmax, p.max())

print(f'Gemma 2 27B (layer {best_layer_d}) — 2-D Neumann overlay:')
axes_d = []
for col, (m, n) in enumerate(overlay_modes):
    ax = fig_d.add_subplot(gs_d[0, col])
    axes_d.append(ax)

    Z = neumann_2d_mode(U, V, m, n)
    ax.contourf(U, V, Z, levels=40, cmap=bg_cmap, vmin=-1, vmax=1)
    c_levels = np.linspace(-1, 1, 11)
    c_levels = c_levels[c_levels != 0]
    ax.contour(U, V, Z, levels=c_levels, colors=['#7A716A'],
               linewidths=0.6, linestyles='--', alpha=0.4)

    pred = np.array(mr_best_d[(m, n)]['pred'])
    r2v  = mr_best_d[(m, n)]['r2']
    pcw  = np.array(mr_best_d[(m, n)]['pc_weights'])
    top3 = np.argsort(np.abs(pcw))[::-1][:3]

    ax.scatter(llm_valence_d, llm_arousal_d, c=pred, cmap='RdBu_r',
               vmin=-shared_vmax, vmax=shared_vmax, s=55, zorder=5,
               edgecolors='white', lw=0.5)
    for i, w in enumerate(llm_words_d):
        ax.annotate(w, (llm_valence_d[i], llm_arousal_d[i]), fontsize=5.5,
                    color=TEXT, ha='center', va='bottom',
                    xytext=(0, 4), textcoords='offset points')

    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_aspect('equal')
    ax.set_title(f'mode ({m},{n})', fontsize=10)
    ax_style(ax, xlabel='valence $u$', ylabel='arousal $v$', grid=False)

    pc_str = ', '.join([f'φ{j+1}({pcw[j]:+.3f})' for j in top3])
    print(f'  mode ({m},{n}): R² = {r2v:.3f}  top PCs: {pc_str}')

# Create colorbar in dedicated subplot to match plot heights
cbar_ax = fig_d.add_subplot(gs_d[0, 2])
cbar_ax.set_position([cbar_ax.get_position().x0 - 0.04, cbar_ax.get_position().y0, 
                      cbar_ax.get_position().width, cbar_ax.get_position().height])
sm = plt.cm.ScalarMappable(cmap='RdBu_r',
                           norm=plt.Normalize(-shared_vmax, shared_vmax))
cbar = fig_d.colorbar(sm, cax=cbar_ax)
cbar.set_label('PC projection', fontsize=9, color=TEXT)
cbar.ax.tick_params(colors=TEXT, labelsize=8)
cbar.outline.set_edgecolor(BORDER)

# Force colorbar to match the height of the main plots
fig_d.canvas.draw()
ref_pos = axes_d[0].get_position()
cbar_pos = cbar_ax.get_position()
cbar_ax.set_position([cbar_pos.x0, ref_pos.y0, cbar_pos.width, ref_pos.height])

# Row label
pos = axes_d[0].get_position()
fig_d.text(pos.x0 - 0.052, (pos.y0 + pos.y1) / 2, 'Gemma 2 27B',
           color=TEXT, fontsize=11, fontweight='bold',
           va='center', ha='right', rotation=90)

fig_d.patch.set_facecolor(BG)
path = FIGS_DIR / 'neumann_2d_main.png'
plt.savefig(path, bbox_inches='tight', facecolor=BG, dpi=400)
plt.show()
print(f'Saved {path.name}')